# 16.6 图卷积网络 GCN / Graph Convolutional Network

**中文**：上一节的 DeepWalk/node2vec 是"浅层"嵌入——只用结构、不用节点特征、且新节点要重训。本节进入**图神经网络(GNN)** 的核心：**GCN(Kipf & Welling, 2017)**。它把深度学习的"卷积"思想搬到图上——**每个节点不断聚合邻居的信息来更新自己的表示**，同时**端到端地融合节点特征**。GCN 是整个 GNN 领域的奠基之作。
**English**: DeepWalk/node2vec were "shallow" embeddings — structure-only, feature-free, and require retraining for new nodes. Now the core of **Graph Neural Networks (GNNs)**: **GCN (Kipf & Welling, 2017)**. It brings deep learning's "convolution" to graphs — **each node repeatedly aggregates its neighbors' information to update its own representation**, while **fusing node features end-to-end**. GCN is the founding work of the GNN field.

---

**中文**：核心思想是**消息传递(message passing)**：一层 GCN 让每个节点"听取所有邻居说什么(聚合)，再更新自己"。第 $l+1$ 层：
**English**: The core idea is **message passing**: one GCN layer lets each node "listen to all neighbors (aggregate), then update itself." Layer $l+1$:

$$H^{(l+1)} = \sigma\!\Big(\hat A\,H^{(l)}\,W^{(l)}\Big),\qquad \hat A = \tilde D^{-1/2}\,\tilde A\,\tilde D^{-1/2},\quad \tilde A = A+I$$

**中文**：逐项解释：
**English**: Term by term:
- **$H^{(l)}$**：第 $l$ 层所有节点的表示矩阵($H^{(0)}=X$ 是输入特征)。
  $H^{(l)}$: all nodes' representations at layer $l$ ($H^{(0)}=X$ is the input features).
- **$\tilde A=A+I$**：邻接矩阵**加自环**——聚合时不能忘了自己。
  $\tilde A=A+I$: adjacency **with self-loops** — a node must include itself when aggregating.
- **$\hat A=\tilde D^{-1/2}\tilde A\tilde D^{-1/2}$**：**对称归一化**(用度开方归一)。为什么？高度节点的邻居很多，直接求和会让它的数值越滚越大；归一化把每条消息按两端度数缩放，**防止数值爆炸、稳定训练**。
  $\hat A$: **symmetric normalization** (by square-root of degree). Why? High-degree nodes have many neighbors; plain summation lets their magnitudes blow up. Normalization scales each message by both endpoints' degrees, **preventing explosion and stabilizing training**.
- **$W^{(l)}$**：可学习的线性变换(像普通神经网络的权重)；$\sigma$ 是激活(ReLU)。
  $W^{(l)}$: a learnable linear transform (like a normal NN weight); $\sigma$ is the activation (ReLU).

**中文**：直觉:$\hat A H W$ 三件事——$W$ 变换特征、$\hat A\cdot$ 做"邻居加权平均"(把每个节点的表示换成它和邻居的平滑混合)、$\sigma$ 加非线性。**堆 $L$ 层 = 每个节点能看到 $L$ 跳以内的邻居**。
**English**: Intuition: $\hat A H W$ does three things — $W$ transforms features, $\hat A\cdot$ performs a "neighbor weighted-average" (replacing each node's representation with a smooth mix of itself and its neighbors), $\sigma$ adds nonlinearity. **Stacking $L$ layers = each node sees its $L$-hop neighborhood**.

> 💡 **面试速查 / Interview cheat-sheet（★★★ GNN 头号必考）**
> **中文**：**GCN = 归一化邻接 × 特征 × 权重 + 激活**，本质是**消息传递/邻居聚合**。$\hat A=\tilde D^{-1/2}(A{+}I)\tilde D^{-1/2}$:加自环(别忘自己)+对称归一(防爆炸)。**半监督节点分类**:只标注少量节点(Cora 每类 20 个), 靠图把标签信息"传播"到未标注节点。**关键认知**:GNN 同时用**结构+特征**, 所以远胜只用特征的 MLP。**头号坑——过平滑(over-smoothing)**:层数太多→所有节点表示趋同→性能崩(2-3 层最佳)。GCN 是**直推式**(需全图), 改进见 GraphSAGE(归纳)、GAT(注意力)。
> **English**: **GCN = normalized-adjacency × features × weights + activation**, essentially **message passing / neighbor aggregation**. $\hat A=\tilde D^{-1/2}(A{+}I)\tilde D^{-1/2}$: self-loops (don't forget yourself) + symmetric normalization (prevent explosion). **Semi-supervised node classification**: label only a few nodes (20/class on Cora) and let the graph "propagate" label info to unlabeled nodes. **Key insight**: GNNs use **structure + features**, so they far exceed a feature-only MLP. **#1 pitfall — over-smoothing**: too many layers → all node representations converge → performance collapses (2–3 layers is best). GCN is **transductive** (needs the whole graph); improvements: GraphSAGE (inductive), GAT (attention).


In [ ]:

# ============================================================
# 数据 Cora + 构造归一化邻接 Â / Cora + normalized adjacency
# ============================================================
import os, time, numpy as np, torch, torch.nn as nn, torch.nn.functional as F, matplotlib.pyplot as plt
torch.manual_seed(0); np.random.seed(0)
R=os.path.expanduser("~/.cache/dsfs_recsys/cora")
content=[l.split("\t") for l in open(os.path.join(R,"cora.content")).read().strip().split("\n")]
ids=[c[0] for c in content]; id2x={v:i for i,v in enumerate(ids)}; n=len(ids)
classes=sorted(set(c[-1] for c in content)); lab2y={c:i for i,c in enumerate(classes)}
y=torch.tensor([lab2y[c[-1]] for c in content])
X=torch.tensor(np.array([[int(x) for x in c[1:-1]] for c in content],dtype=np.float32))
X=X/X.sum(1,keepdim=True).clamp(min=1)                       # 行归一化特征 / row-normalize features

A=np.zeros((n,n),dtype=np.float32)                           # 邻接矩阵 / adjacency
for line in open(os.path.join(R,"cora.cites")).read().strip().split("\n"):
    a,b=line.split("\t")
    if a in id2x and b in id2x: A[id2x[a],id2x[b]]=1; A[id2x[b],id2x[a]]=1
A=A+np.eye(n,dtype=np.float32)                               # 加自环 Ã=A+I / add self-loops
d=A.sum(1); Dinv=1.0/np.sqrt(d)                              # 度的 -1/2 次方 / D^{-1/2}
Ahat=torch.tensor(Dinv[:,None]*A*Dinv[None,:])              # 对称归一化 Â / symmetric normalization

# 半监督划分: 每类 20 个训练, 500 验证, 1000 测试 / semi-supervised split
np.random.seed(0); train_mask=np.zeros(n,bool)
for c in range(len(classes)):
    idx=np.where(y.numpy()==c)[0]; np.random.shuffle(idx); train_mask[idx[:20]]=True
rest=np.where(~train_mask)[0]; np.random.shuffle(rest)
val_mask=np.zeros(n,bool); test_mask=np.zeros(n,bool); val_mask[rest[:500]]=True; test_mask[rest[500:1500]]=True
tm,vm,tem=torch.tensor(train_mask),torch.tensor(val_mask),torch.tensor(test_mask)
print(f"节点 {n}, 特征 {X.shape[1]}, 类别 {len(classes)}")
print(f"标注训练节点 {tm.sum().item()} (仅 {tm.float().mean()*100:.0f}%!), 验证 {vm.sum().item()}, 测试 {tem.sum().item()}")
print("→ 半监督:只用极少标签, 靠图结构传播 / semi-supervised: tiny labels, propagate via graph")


**中文**：从零实现一个 **2 层 GCN**：第一层把 1433 维特征聚合+变换到 16 维隐藏表示(ReLU)，第二层聚合+变换到 7 维(类别 logits)。同时实现一个**结构盲的 MLP**(同样的层，但**不乘 $\hat A$**，即不看图)作为对照——看图结构到底带来多大提升。
**English**: Implement a **2-layer GCN** from scratch: layer 1 aggregates+transforms the 1433-dim features to a 16-dim hidden representation (ReLU); layer 2 aggregates+transforms to 7-dim (class logits). We also implement a **structure-blind MLP** (same layers but **without multiplying by $\hat A$**, i.e. ignoring the graph) as a control — to see exactly how much the graph structure helps.


In [ ]:

# ============================================================
# 从零实现 GCN 与对照 MLP / GCN and control MLP from scratch
# ============================================================
class GCN(nn.Module):
    def __init__(s,fin,h,fout):
        super().__init__(); s.W0=nn.Linear(fin,h); s.W1=nn.Linear(h,fout); s.dp=nn.Dropout(0.5)
    def forward(s,X,Ahat):
        H=F.relu(Ahat @ s.W0(s.dp(X)))        # 第1层:聚合邻居+变换+ReLU / aggregate+transform+relu
        return Ahat @ s.W1(s.dp(H))           # 第2层:再聚合+变换 -> logits / again -> logits

class MLP(nn.Module):                          # 同样结构但不乘 Â (不看图) / same but no graph
    def __init__(s,fin,h,fout):
        super().__init__(); s.W0=nn.Linear(fin,h); s.W1=nn.Linear(h,fout); s.dp=nn.Dropout(0.5)
    def forward(s,X,Ahat):
        return s.W1(s.dp(F.relu(s.W0(s.dp(X)))))

def train_model(Model, epochs=200):
    torch.manual_seed(0); m=Model(X.shape[1],16,len(classes))
    opt=torch.optim.Adam(m.parameters(),lr=0.01,weight_decay=5e-4)
    best_val=0; best_test=0; curve=[]
    for ep in range(epochs):
        m.train(); opt.zero_grad()
        loss=F.cross_entropy(m(X,Ahat)[tm], y[tm])             # 只在标注节点上算损失 / loss on labeled only
        loss.backward(); opt.step()
        m.eval()
        with torch.no_grad():
            pred=m(X,Ahat).argmax(1)
            va=(pred[vm]==y[vm]).float().mean().item(); ta=(pred[tem]==y[tem]).float().mean().item()
            curve.append(ta)
            if va>best_val: best_val, best_test = va, ta        # 按验证集选最佳 / select by val
    return best_test, curve, m

gcn_acc, gcn_curve, gcn_model = train_model(GCN)
mlp_acc, mlp_curve, _ = train_model(MLP)
print(f"GCN 测试准确率 / test accuracy: {gcn_acc:.4f}")
print(f"MLP 测试准确率(不看图) / MLP (graph-blind): {mlp_acc:.4f}")
print(f"→ 图结构带来 +{(gcn_acc-mlp_acc)*100:.1f} 个百分点 / graph structure adds {(gcn_acc-mlp_acc)*100:.0f} points!")


**中文**：GCN 显著超过 MLP——**同样的特征、同样的标签、同样的网络容量，唯一区别是 GCN 用了图结构**(乘 $\hat A$ 聚合邻居)。这干净地证明了"邻居信息"的价值：在半监督设定下(只有 5% 节点有标签)，标签信息能沿着引用边"传播"到未标注节点。

下面做 GNN **最重要的诚实实验——过平滑(over-smoothing)**:堆更多层理论上能看更远的邻居，但实际会发生什么？
**English**: GCN clearly beats the MLP — **same features, same labels, same capacity; the only difference is GCN uses the graph structure** (multiplying by $\hat A$ to aggregate neighbors). This cleanly proves the value of "neighbor information": under the semi-supervised setting (only 5% labeled), label info "propagates" along citation edges to unlabeled nodes.

Now the **most important honest GNN experiment — over-smoothing**: stacking more layers should in theory see farther neighbors, but what actually happens?


In [ ]:

# ============================================================
# 过平滑实验: 层数越深越糟 / over-smoothing: deeper is worse
# ============================================================
class DeepGCN(nn.Module):
    def __init__(s,fin,h,fout,L):
        super().__init__()
        dims=[fin]+[h]*(L-1)+[fout]
        s.layers=nn.ModuleList([nn.Linear(dims[i],dims[i+1]) for i in range(L)])   # L 层 / L layers
        s.dp=nn.Dropout(0.5)
    def forward(s,X,Ahat):
        H=X
        for i,lin in enumerate(s.layers):
            H=Ahat @ lin(s.dp(H))                                # 每层都聚合一次 / aggregate each layer
            if i<len(s.layers)-1: H=F.relu(H)
        return H

def train_depth(L, epochs=200):
    torch.manual_seed(0); m=DeepGCN(X.shape[1],16,len(classes),L)
    opt=torch.optim.Adam(m.parameters(),lr=0.01,weight_decay=5e-4); bv=0; bt=0
    for ep in range(epochs):
        m.train(); opt.zero_grad(); F.cross_entropy(m(X,Ahat)[tm],y[tm]).backward(); opt.step()
        m.eval()
        with torch.no_grad():
            pred=m(X,Ahat).argmax(1); va=(pred[vm]==y[vm]).float().mean().item(); ta=(pred[tem]==y[tem]).float().mean().item()
            if va>bv: bv,bt=va,ta
    return bt

depths=[2,3,4,8,16]; depth_acc=[train_depth(L) for L in depths]
print(f"{'层数 L':<8}{'测试准确率 test acc':>20}")
for L,a in zip(depths,depth_acc): print(f"{L:<8}{a:>20.4f}")
print(f"\n随机猜测基线 / random baseline = 1/{len(classes)} = {1/len(classes):.3f}")


In [ ]:

# ============================================================
# 可视化 / Visualization
# ============================================================
from sklearn.manifold import TSNE
fig,ax=plt.subplots(1,3,figsize=(17,5))
# ① GCN vs MLP 测试准确率曲线 / accuracy curves
ax[0].plot(gcn_curve,label=f"GCN (best {gcn_acc:.3f})",color="#4C72B0")
ax[0].plot(mlp_curve,label=f"MLP 不看图 (best {mlp_acc:.3f})",color="#C44E52")
ax[0].set_title("GCN vs MLP:图结构的价值 / value of structure"); ax[0].set_xlabel("epoch"); ax[0].set_ylabel("test acc"); ax[0].legend()
# ② 过平滑:层数 vs 准确率 / over-smoothing
ax[1].plot(depths,depth_acc,"o-",color="#55A868",ms=9)
ax[1].axhline(1/len(classes),ls="--",color="gray",label=f"随机 random {1/len(classes):.2f}")
ax[1].set_title("过平滑:层越多越糟 / over-smoothing"); ax[1].set_xlabel("层数 #layers L"); ax[1].set_ylabel("test acc"); ax[1].legend()
# ③ GCN 学到的隐表示 t-SNE / t-SNE of GCN hidden representation
gcn_model.eval()
with torch.no_grad():
    H=F.relu(Ahat @ gcn_model.W0(X)).numpy()                 # 第一层后的隐表示 / hidden after layer 1
Z=TSNE(n_components=2,init="pca",random_state=0,perplexity=30).fit_transform(H)
pal=plt.cm.tab10(np.linspace(0,1,len(classes)))
for c in range(len(classes)): mk=(y.numpy()==c); ax[2].scatter(Z[mk,0],Z[mk,1],s=8,color=pal[c],alpha=0.7)
ax[2].set_title("GCN 隐表示(t-SNE):类别分离 / hidden separates classes"); ax[2].set_xticks([]); ax[2].set_yticks([])
plt.tight_layout(); plt.savefig("/tmp/g06_viz.png",dpi=80); plt.show()
print("深层 GCN 准确率跌向随机线 = 过平滑 / deep GCN collapses toward random = over-smoothing")


**中文**：诚实解读：
**English**: Honest takeaways:

**中文**：
1. **图结构带来巨大提升**：GCN(~0.77)远超结构盲的 MLP(~0.57)——足足 **+20 个百分点**，而两者特征、标签、容量完全相同。这是 GNN 价值最干净的证明:**在标签稀缺时，图让标签信息沿边传播**。
2. **过平滑是 GNN 的头号陷阱**：层数从 2 增到 16，准确率从 0.77 一路崩到 ~0.15(≈随机猜测 1/7)。原因:每多一层就多做一次"邻居平滑"，层数过多后**所有节点的表示被反复平均到几乎一样**，再也区分不开。所以 GCN **通常只用 2~3 层**——这与 CNN"越深越好"完全相反，是图深度学习独有的反直觉现象。
3. **t-SNE 显示** GCN 的隐表示把 7 个类别分得很开——结构+特征联合学习的成果。

**English**:
1. **Graph structure gives a huge boost**: GCN (~0.77) far exceeds the structure-blind MLP (~0.57) — a full **+20 points**, with identical features, labels, and capacity. The cleanest proof of GNN value: **when labels are scarce, the graph propagates label info along edges**.
2. **Over-smoothing is the #1 GNN pitfall**: going from 2 to 16 layers, accuracy collapses from 0.77 to ~0.15 (≈ random 1/7). Reason: each extra layer does another "neighbor smoothing"; with too many, **all node representations get repeatedly averaged into near-identical vectors** and become indistinguishable. So GCN **usually uses just 2–3 layers** — the opposite of CNNs' "deeper is better," a counter-intuitive phenomenon unique to graph deep learning.
3. **t-SNE shows** GCN's hidden representation separates the 7 classes well — the fruit of jointly learning from structure + features.

> 💼 **实战视角 / Practical angle**
> **中文**:GCN 用于:节点分类(论文/用户分类)、推荐(用户-物品图)、欺诈检测、分子性质预测、交通预测。**工程要点**:① 大图无法把 $\hat A$ 整个放内存→用稀疏矩阵 + **邻居采样**(下节 GraphSAGE); ② 层数 2-3, 想加深要用**残差/跳连/PairNorm/DropEdge** 缓解过平滑; ③ GCN 是**直推式**(训练要全图、新节点要重算)。面试金句:*"GCN 就是带归一化邻接的消息传递; 它同时吃结构和特征所以胜过 MLP; 但层数不能多——过平滑会让所有节点变得一样。"*
> **English**: GCN is used for: node classification (papers/users), recommendation (user-item graphs), fraud detection, molecular property prediction, traffic forecasting. **Engineering**: ① huge graphs can't hold $\hat A$ in memory → sparse matrices + **neighbor sampling** (GraphSAGE, next); ② use 2–3 layers, and to go deeper apply **residual/skip connections / PairNorm / DropEdge** to fight over-smoothing; ③ GCN is **transductive** (needs the full graph at training; new nodes require recomputation). Interview line: *"GCN is message passing with a normalized adjacency; it uses both structure and features so it beats an MLP; but you can't stack many layers — over-smoothing makes all nodes look alike."*

---
### 小结 / Summary
- **中文**:GCN=$\sigma(\hat A H W)$,本质邻居聚合(消息传递);$\hat A$=加自环+对称归一化。
- **English**: GCN = $\sigma(\hat A H W)$, essentially neighbor aggregation (message passing); $\hat A$ = self-loops + symmetric normalization.
- **中文**:半监督下 GCN(用结构+特征)远胜 MLP(只用特征)——图传播了标签信息。
- **English**: Semi-supervised, GCN (structure + features) far beats MLP (features only) — the graph propagates label info.
- **中文**:头号坑过平滑——层数过多节点表示趋同, 通常只用 2~3 层。
- **English**: #1 pitfall over-smoothing — too many layers make node representations converge; use only 2–3 layers.
